# Baseline BM25 --- Notebook de apoio

**Disciplina:** Inteligência Artificial --- FACOM/UFMS --- 2026/1

Este notebook carrega o `corpus.jsonl` gerado pelo notebook de coleta (`coleta_arxiv.ipynb`), treina um índice **BM25** e devolve o *top-k* para uma consulta-exemplo. É o seu **baseline obrigatório**. A partir daqui, você só precisa adicionar pré-processamento mais cuidadoso, comparar com KNN/denso e implementar os módulos de aprofundamento.

**Atenção:** este código é propositalmente "mínimo viável". Você é avaliado também pelo **rigor das suas escolhas** (justificativa dos hiperparâmetros, pré-processamento, conexão com o paradigma probabilístico de Naïve Bayes). Não entregue isto como está --- evolua-o.

In [ ]:
# !pip install rank_bm25 nltk pandas

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
from rank_bm25 import BM25Okapi

# stopwords do NLTK (somente uma vez)
import nltk
try:
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    from nltk.corpus import stopwords
    STOP = set(stopwords.words("english"))

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/jeanverso/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


## 1. Carregamento do corpus

Ajuste o caminho se o seu `corpus.jsonl` estiver em outro lugar.

In [3]:
CORPUS_PATH = Path("../../data/corpus.jsonl")  # ajuste se necessário

docs = []
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        docs.append(json.loads(line))

print(f"Documentos carregados: {len(docs)}")
docs[0]

Documentos carregados: 1808


{'arxiv_id': '1811.08390',
 'title': 'Structured Pruning for Efficient ConvNets via Incremental Regularization',
 'abstract': 'Parameter pruning is a promising approach for CNN compression and acceleration by eliminating redundant model parameters with tolerable performance loss. Despite its effectiveness, existing regularization-based parameter pruning methods usually drive weights towards zero with large and constant regularization factors, which neglects the fact that the expressiveness of CNNs is fragile and needs a more gentle way of regularization for the networks to adapt during pruning. To solve this problem, we propose a new regularization-based pruning method (named IncReg) to incrementally assign different regularization factors to different weight groups based on their relative importance, whose effectiveness is proved on popular CNNs compared with state-of-the-art methods.',
 'authors': ['Huan Wang', 'Qiming Zhang', 'Yuehai Wang', 'Haoji Hu'],
 'categories': ['cs.CV', 'cs.

## 2. Pré-processamento

Mínimo aceitável: *lower-casing*, remoção de pontuação, remoção de *stopwords*. Não usamos *stemming* aqui para não esconder discussão (você pode adicionar e comparar).

In [5]:
TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z\-]+")

def tokenize(text: str):
    text = text.lower()
    tokens = TOKEN_RE.findall(text)
    return [t for t in tokens if t not in STOP and len(t) > 2]

tokenize("Parameter pruning is a promising approach for CNN compression and acceleration by eliminating redundant model parameters with tolerable performance loss")

['parameter',
 'pruning',
 'promising',
 'approach',
 'cnn',
 'compression',
 'acceleration',
 'eliminating',
 'redundant',
 'model',
 'parameters',
 'tolerable',
 'performance',
 'loss']

In [6]:
# Texto indexado = título + abstract
texts = [(d["title"] + ". " + d["abstract"]) for d in docs]
tokenized_corpus = [tokenize(t) for t in texts]
print("Exemplo de tokens do doc 0:", tokenized_corpus[0][:20])

Exemplo de tokens do doc 0: ['structured', 'pruning', 'efficient', 'convnets', 'via', 'incremental', 'regularization', 'parameter', 'pruning', 'promising', 'approach', 'cnn', 'compression', 'acceleration', 'eliminating', 'redundant', 'model', 'parameters', 'tolerable', 'performance']


## 3. Índice BM25

Hiperparâmetros padrão do `rank_bm25`: $k_1=1.5$, $b=0.75$. Documente no seu relatório qual valor adotou e por quê. O módulo M4 (otimização) pode ajustá-los empiricamente.

In [7]:
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print("Vocabulário BM25:", len(bm25.idf), "termos")

Vocabulário BM25: 15729 termos


## 4. Função de busca

In [8]:
def search(query: str, k: int = 10):
    q_tokens = tokenize(query)
    scores = bm25.get_scores(q_tokens)
    top_idx = scores.argsort()[::-1][:k]
    return [(int(i), float(scores[i]), docs[i]) for i in top_idx]

In [9]:
query = "Structured Pruning for Efficient ConvNets via Incremental Regularization"
results = search(query, k=10)

for rank, (idx, score, d) in enumerate(results, 1):
    print(f"{rank:>2}. [{score:6.2f}] {d['title']}")
    print(f"     {d['arxiv_id']} | {d.get('primary_category')} | {d.get('published','')[:10]}")
    print(f"     {d['abstract'][:200]}...")
    print()

 1. [ 25.22] Structured Pruning for Efficient ConvNets via Incremental Regularization
     1811.08390 | cs.CV | 2018-11-20
     Parameter pruning is a promising approach for CNN compression and acceleration by eliminating redundant model parameters with tolerable performance loss. Despite its effectiveness, existing regulariza...

 2. [ 23.30] Structured Pruning for Efficient ConvNets via Incremental Regularization
     1804.09461 | cs.LG | 2018-04-25
     Parameter pruning is a promising approach for CNN compression and acceleration by eliminating redundant model parameters with tolerable performance degrade. Despite its effectiveness, existing regular...

 3. [ 11.72] One-Cycle Structured Pruning via Stability-Driven Subnetwork Search
     2501.13439 | cs.CV | 2025-01-23
     Existing structured pruning methods typically rely on multi-stage training procedures that incur high computational costs. Pruning at initialization aims to reduce this burden but often suffers from d...

 4. [ 

## 5. Gerando um arquivo `runs/bm25.trec` para avaliação

Formato TREC tradicional: `qid Q0 doc_id rank score system`. Esse formato é aceito por `pytrec_eval` e `ir_measures`, e é o que o script `evaluate.py` (no diretório `eval/`) consome.

In [11]:
queries_path = Path("./eval/queries.tsv")  # qid<TAB>texto da query
runs_dir = Path("./runs"); runs_dir.mkdir(exist_ok=True)
run_path = runs_dir / "bm25.trec"

queries = pd.read_csv(queries_path, sep="\t", names=["qid", "text"])
with open(run_path, "w", encoding="utf-8") as f:
    for _, q in queries.iterrows():
        for rank, (idx, score, d) in enumerate(search(q["text"], k=100), 1):
            f.write(f"{q['qid']} Q0 {d['arxiv_id']} {rank} {score:.6f} bm25\n")

print("Run salva em:", run_path)
!head -n 5 {run_path}

Run salva em: runs/bm25.trec
q1 Q0 2606.00257 1 13.567650 bm25
q1 Q0 2508.09715 2 12.954739 bm25
q1 Q0 2411.16155 3 12.933483 bm25
q1 Q0 2507.09132 4 11.548524 bm25
q1 Q0 2308.02565 5 11.011658 bm25


## 6. Próximos passos

1. Substitua o tokenizador por algo mais robusto (e.g., spaCy) e compare.
2. Implemente um segundo *retriever* (KNN + TF-IDF, ou KNN + *embeddings*) e gere `runs/knn.trec`.
3. Construa as `qrels` (Seção 2 do template em `eval/qrels_example.tsv`).
4. Use `eval/evaluate.py` para comparar.
5. Implemente o(s) módulo(s) de aprofundamento e gere mais arquivos de *run*.
6. Escreva tudo no relatório SBC. ✍️